# ML-05 — Feature Vector and Leakage/Privacy Check

Deep audit of feature availability, missingness, and leakage prevention for the Content Refresh Opportunity lane.

## 1. Build the feature vector

We construct observable pre-decision features from historical 90-day search logs.

In [1]:
import pandas as pd
import numpy as np
import os

data_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "/content/FlyRank-Internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Filter for active search-visible pages
active_df = df[df["avg_position"] > 0].copy()

# Feature engineering
active_df["log_search_volume"] = np.log1p(active_df["search_volume"].fillna(0))
active_df["log_impressions_90d"] = np.log1p(active_df["impressions_90d"].fillna(0))
active_df["log_clicks_90d"] = np.log1p(active_df["clicks_90d"].fillna(0))
active_df["ctr_observed"] = active_df["ctr"].fillna(0)
active_df["avg_position_observed"] = active_df["avg_position"]
active_df["days_since_update"] = active_df["days_since_last_update"].fillna(0)
active_df["recent_impression_change"] = (active_df["impressions_last_30d"] - active_df["impressions_prev_30d"]).fillna(0)

features = [
    "log_search_volume", "log_impressions_90d", "log_clicks_90d",
    "ctr_observed", "avg_position_observed", "days_since_update", "recent_impression_change"
]

X = active_df[features]
print("Feature matrix shape:", X.shape)
display(X.head())


Feature matrix shape: (28795, 7)


,log_search_volume,log_impressions_90d,log_clicks_90d,ctr_observed,avg_position_observed,days_since_update,recent_impression_change
0,2.397895,8.243808,3.401197,0.76,10.6,20,-409
1,4.510860,9.636980,2.079442,0.05,20.3,25,-3414
2,0.000000,9.440023,2.484907,0.09,36.5,20,-3707
3,2.397895,9.371779,4.077537,0.49,6.2,22,-580
4,0.000000,9.859588,3.218876,0.13,44.0,14,-2241


## 2. Feature notes (meaning, missing, categorical, available-when?)

Every feature must be knowable before the editorial refresh decision.

In [2]:
feature_metadata = pd.DataFrame([
    {"Feature": "log_search_volume", "Type": "Continuous", "Missing": active_df["search_volume"].isna().sum(), "Available When": "Historical keyword query volume"},
    {"Feature": "log_impressions_90d", "Type": "Continuous", "Missing": active_df["impressions_90d"].isna().sum(), "Available When": "Observed trailing 90d search visibility"},
    {"Feature": "log_clicks_90d", "Type": "Continuous", "Missing": active_df["clicks_90d"].isna().sum(), "Available When": "Observed trailing 90d search clicks"},
    {"Feature": "ctr_observed", "Type": "Percentage", "Missing": active_df["ctr"].isna().sum(), "Available When": "Observed trailing 90d CTR"},
    {"Feature": "avg_position_observed", "Type": "Continuous", "Missing": active_df["avg_position"].isna().sum(), "Available When": "Observed trailing 90d average ranking"},
    {"Feature": "days_since_update", "Type": "Continuous", "Missing": active_df["days_since_last_update"].isna().sum(), "Available When": "CMS metadata timestamp"},
    {"Feature": "recent_impression_change", "Type": "Continuous", "Missing": 0, "Available When": "Pre-decision 30d vs previous 30d delta"}
])
display(feature_metadata)


,Feature,Type,Missing,Available When
0,log_search_volume,Continuous,1719,Historical keyword query volume
1,log_impressions_90d,Continuous,0,Observed trailing 90d search visibility
2,log_clicks_90d,Continuous,0,Observed trailing 90d search clicks
3,ctr_observed,Percentage,0,Observed trailing 90d CTR
4,avg_position_observed,Continuous,0,Observed trailing 90d average ranking
5,days_since_update,Continuous,0,CMS metadata timestamp
6,recent_impression_change,Continuous,0,Pre-decision 30d vs previous 30d delta


## 3. The leakage hunt

Verify that none of the features correlate 1:1 with target labels or contain outcome definitions.

In [3]:
# Define ground-truth target proxy (downward performance trend)
y = active_df["trend_direction"].str.lower().eq("down").astype(int)

# Check correlation of each feature with the label
correlations = X.apply(lambda col: np.corrcoef(col, y)[0, 1])
corr_df = pd.DataFrame({"Feature": features, "Correlation_with_Label": correlations.round(4)})
display(corr_df)

# Assert no perfect correlation (no feature leakage)
assert (correlations.abs() < 0.95).all(), "Leakage detected: Feature correlation too high!"
print("PASS: No feature leakage detected. Max absolute correlation is:", round(correlations.abs().max(), 4))


,Feature,Correlation_with_Label
log_search_volume,log_search_volume,-0.0844
log_impressions_90d,log_impressions_90d,0.1011
log_clicks_90d,log_clicks_90d,-0.0344
ctr_observed,ctr_observed,-0.0688
avg_position_observed,avg_position_observed,-0.0813
days_since_update,days_since_update,0.0532
recent_impression_change,recent_impression_change,-0.1784


PASS: No feature leakage detected. Max absolute correlation is: 0.1784


## 4. What I excluded and why

Strict exclusion list to prevent data leakage and target contamination.

In [4]:
excluded_fields = pd.DataFrame([
    {"Excluded Field": "trend_pct", "Reason": "Directly derives the trend_direction label (target contamination)"},
    {"Excluded Field": "trend_direction", "Reason": "This is the evaluation target label itself"},
    {"Excluded Field": "client_id / content_id", "Reason": "Pseudonymized identifiers used for grouping and splits only, never features"},
    {"Excluded Field": "Future window metrics", "Reason": "Unavailable at the decision snapshot moment"}
])
display(excluded_fields)


,Excluded Field,Reason
0,trend_pct,Directly derives the trend_direction label (ta...
1,trend_direction,This is the evaluation target label itself
2,client_id / content_id,Pseudonymized identifiers used for grouping an...
3,Future window metrics,Unavailable at the decision snapshot moment


## Self-check

- [x] Feature vector constructed with zero missing value crashes
- [x] No target-derived features or future windows included
- [x] Executed top-to-bottom with visible outputs